# 08 Robustness of the measure and ablations
Repeats the reference analysis under other thresholds, null tolerances, eccentricity ranges, pRF halves and cortical subdivisions, using the cached tractograms, and summarises the ablation networks.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
import copy
from retinopath import metrics as M
subjects = PL.make_subject_lists(cfg); test = [PL.session(s) for s in subjects["test"]]
variants = {"default": {}, "r2>=5": {"r2_min": 5}, "r2>=20": {"r2_min": 20}, "null tol 5%": {"null_tol": 0.05}, "null tol 20%": {"null_tol": 0.20},
            "ecc<=5": {"ecc_max": 5.0}, "ecc>=1": {"ecc_min": 1.0}}
rows = []
with Timer(cfg, "08_robustness"):
    for ses in test:
        a = PL.load_anat(cfg, ses); t = PL.tract(cfg, ses, "REF", "msmt_csd", anat=a)
        for name, ch in variants.items():
            c = copy.deepcopy(cfg); c.prf.update(ch)
            rows.append(dict(sid=ses["sid"], analysis=name, **M.retinotopic_scores(a, t["a"], t["b"], t["pair"], c, seed=cfg.seed)))
        for fit in (1, 2):                                   # key from one half of the pRF runs only
            r = M.retinotopic_scores(a, t["a"], t["b"], t["pair"], cfg, key=M.get_key(a, cfg, fit), seed=cfg.seed)
            rows.append(dict(sid=ses["sid"], analysis=f"pRF half {fit}", **r))
        for sub, m in (("dorsal", a["dorsal"][t["a"]]), ("ventral", ~a["dorsal"][t["a"]]), ("left", a["hemi"][t["a"]] == 0), ("right", a["hemi"][t["a"]] == 1)):
            if m.sum() > 50:
                rows.append(dict(sid=ses["sid"], analysis=sub, **M.retinotopic_scores(a, t["a"][m], t["b"][m], t["pair"][m], cfg, seed=cfg.seed)))
rob = pd.DataFrame(rows); rob.to_csv(RES / "robustness.csv", index=False)
rob.groupby("analysis", sort=False)[["n_valid", "rsi_ree_all", "rsi_ree_V1-V2", "rsi_ree_V2-V3", "rsi_ree_V1-V3"]].median().round(3)

In [ ]:
# Ablations: networks trained in notebook 05, scored in notebook 06
f = RES / "learned.csv"
if f.exists():
    d = pd.read_csv(f); d["variant"] = d.method.str.extract(r"net_(.*)_s\d+")[0]
    display(d.groupby(["variant", "protocol"])[["rsi_ree_all", "acc", "peak_err_deg", "n_pairs"]].median().round(3).unstack())
else:
    print("run notebook 06 first")